# IGRF-14 vs pygeodyn GOVO reanalysis

Difference between the two main-field models, **GOVO − IGRF**, at a few dates, at Earth's surface and at the core-mantle boundary (CMB).
GOVO is the ensemble mean of `2023_gji_gnm_GOVO.hdf5` (degree 13, 1997-02 to 2022-10); IGRF-14 is degree 13 from 2000 on.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.ticker import MaxNLocator
from samos.shmodel import SHModel
%matplotlib inline

igrf = SHModel.read_sch_igrf()
govo = SHModel.read_pygeodyn('/project/samos/2023_gji_gnm_GOVO/2023_gji_gnm_GOVO.hdf5')

In [ ]:
dates = np.array(['2000-01-01', '2010-01-01', '2015-01-01', '2020-01-01'], dtype = 'datetime64[D]')
radii = {'surface': 6371.2, 'CMB': 3485.0}                     # km
component = 0                                                   # 0: Br, 1: Btheta, 2: Bphi
label = [r'$B_r$', r'$B_\theta$', r'$B_\phi$'][component]

lat = np.arange(-89.5, 90, 1.0); lon = np.arange(-179.5, 180, 1.0)

# one call per model: all dates and both radii -> (Nt, Nr, Nlat, Nlon)
r = list(radii.values())
diff = (govo.calculate_field_grid(dates, r, 90 - lat, lon)[component]
        - igrf.calculate_field_grid(dates, r, 90 - lat, lon)[component])

w = np.cos(np.radians(lat))[:, None]
rms = np.sqrt((diff**2 * w).sum(axis = (-2, -1)) / (w.sum() * len(lon)))
for i, d in enumerate(dates):
    print(f'{d}:  rms difference  ' + '   '.join(f'{name} {rms[i, j]:7.1f} nT' for j, name in enumerate(radii)))

In [ ]:
fig, axs = plt.subplots(len(dates), len(radii), figsize = (11, 2.6 * len(dates)), layout = 'constrained',
                        subplot_kw = {'projection': ccrs.Mollweide()})

for j, name in enumerate(radii):
    vmax = np.abs(diff[:, j]).max()                             # one colour scale per radius, shared by all dates
    for i, d in enumerate(dates):
        ax = axs[i, j]
        ax.set_global()
        ax.coastlines(linewidth = 0.5)
        mesh = ax.pcolormesh(lon, lat, diff[i, j], transform = ccrs.PlateCarree(),
                             cmap = 'RdBu_r', vmin = -vmax, vmax = vmax)
        ax.set_title(f'{d}, {name} (rms {rms[i, j]:.0f} nT)', fontsize = 10)
    cbar = fig.colorbar(mesh, ax = axs[:, j], orientation = 'horizontal', shrink = 0.7, pad = 0.02)
    cbar.ax.xaxis.set_major_locator(MaxNLocator(6))
    cbar.set_label(f'GOVO − IGRF, {label} [nT]')

plt.show()